# 1.3 Regresión polinómica y regularización


**Objetivo:** Comparar regresión lineal, polinómica, Ridge y Lasso en un mismo problema.  
**Dataset:** `rendimiento_estudiantil.csv`



## Sesión

1. Cargar y revisar el dataset.  
2. Preparar variables de entrada y salida.  
3. Entrenar el modelo.  
4. Evaluar resultados.  
5. Interpretar lo que ocurrió.  
6. Resolver una actividad.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [2]:
from pathlib import Path
import pandas as pd

def cargar_csv(nombre_archivo):
  # Lista de posibles ubicaciones donde puede estar el archivo
    # (funciona tanto en Colab como en Jupyter local)
    rutas = [
        Path(nombre_archivo),   # carpeta actual
        Path("../datasets") / nombre_archivo,  # carpeta datasets arriba
        Path("datasets") / nombre_archivo,  # carpeta datasets aquí
        Path("/content") / nombre_archivo,    # raíz de Colab
        Path("/content/datasets") / nombre_archivo,  # datasets en Colab
    ]
     # Revisa cada ruta hasta encontrar el archivo
    for ruta in rutas:
        if ruta.exists():
            print(f"Archivo encontrado en: {ruta}")
            return pd.read_csv(ruta)
             # Si no lo encontró en ninguna ruta, lanza un error con instrucciones
    raise FileNotFoundError(
        f"No se encontró {nombre_archivo}. Súbelo a Colab o colócalo en la carpeta datasets."
    )


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score


In [4]:
# Cargamos el dataset completo
df = cargar_csv("rendimiento_estudiantil.csv")
# X = todas las columnas EXCEPTO promedio_final (son las variables de entrada)
X = df.drop(columns="promedio_final")
# y = lo que queremos predecir
y = df["promedio_final"]


# Dividimos: 80% para entrenar, 20% para probar
# random_state=42 asegura que la división sea siempre la misma
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# Definimos los 4 modelos a comparar dentro de un diccionario
modelos = {
    # Regresión lineal clásica, sin transformaciones
    "Lineal": Pipeline([
        ("reg", LinearRegression())
    ]),
    # Regresión polinómica: crea combinaciones entre variables (ej: horas² , horas×asistencia)
    "Polinómico grado 2": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("reg", LinearRegression())
    ]),


    # Ridge: penaliza coeficientes grandes pero no los elimina
    # alpha controla qué tan fuerte es la penalización
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),  # Ridge necesita que las variables estén en la misma escala
        ("ridge", Ridge(alpha=1.0))
    ]),
     # Lasso: penaliza coeficientes y puede llevar algunos exactamente a cero
    # (elimina variables poco importantes automáticamente)
    "Lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(alpha=0.2))
    ])
}
# Entrenamos cada modelo y guardamos sus métricas
resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)   # Entrenamos con datos de entrenamiento
    pred = modelo.predict(X_test)   # Predecimos con datos de prueba (nuevos)
    resultados.append({
        "modelo": nombre,
        "RMSE_test": np.sqrt(mean_squared_error(y_test, pred)),   # Error promedio (menor = mejor)
        "R2_test": r2_score(y_test, pred)                # Qué tanto explica el modelo (mayor = mejor)

    })
# Mostramos la tabla ordenada del mejor al peor modelo según RMSE
pd.DataFrame(resultados).round(4).sort_values("RMSE_test")


Archivo encontrado en: rendimiento_estudiantil.csv


,modelo,RMSE_test,R2_test
1,Polinómico grado 2,2.9747,0.4058
3,Lasso,3.2952,0.2708
2,Ridge,3.3289,0.2559
0,Lineal,3.3301,0.2553


## ¿Qué estamos comparando?

- **Lineal:** usa una relación directa entre entradas y salida.
- **Polinómico:** crea combinaciones y curvaturas.
- **Ridge:** penaliza coeficientes grandes y ayuda a estabilizar.
- **Lasso:** además puede llevar algunos coeficientes a cero.


In [5]:
# Volvemos a entrenar Ridge para extraer sus coeficientes
modelo_ridge = modelos["Ridge"]
modelo_ridge.fit(X_train, y_train)

# Creamos una tabla que muestra cuánto "pesa" cada variable en Ridge
# Los coeficientes son pequeños pero ninguno llega exactamente a cero
coeficientes_ridge = pd.DataFrame({
    "variable": X.columns,
    "coeficiente_ridge": np.round(modelo_ridge.named_steps["ridge"].coef_, 4)
})

coeficientes_ridge


,variable,coeficiente_ridge
0,horas_estudio,1.9476
1,asistencia_pct,0.4722
2,tareas_entregadas,1.0361
3,participacion,0.8876
4,horas_sueno,-0.2624


In [6]:
# Volvemos a entrenar Lasso para extraer sus coeficientes
modelo_lasso = modelos["Lasso"]
modelo_lasso.fit(X_train, y_train)

# Creamos una tabla similar pero con Lasso
# Aquí sí verás algunos coeficientes = 0.0 (esas variables fueron eliminadas)
coeficientes_lasso = pd.DataFrame({
    "variable": X.columns,
    "coeficiente_lasso": np.round(modelo_lasso.named_steps["lasso"].coef_, 4)
})

coeficientes_lasso


,variable,coeficiente_lasso
0,horas_estudio,1.7135
1,asistencia_pct,0.2478
2,tareas_entregadas,0.7920
3,participacion,0.6663
4,horas_sueno,-0.0623


## Actividad

1. Cambia `alpha` de Ridge a `0.1`, `1.0` y `10.0`.  
2. Cambia `alpha` de Lasso a `0.05`, `0.2` y `0.8`.  
3. Observa qué ocurre con las métricas y con los coeficientes.


In [7]:
# Espacio para experimentar con distintos valores de alpha


# Cargamos el dataset completo
df = cargar_csv("rendimiento_estudiantil.csv")
# X = todas las columnas EXCEPTO promedio_final (son las variables de entrada)
X = df.drop(columns="promedio_final")
# y = lo que queremos predecir
y = df["promedio_final"]


# Dividimos: 80% para entrenar, 20% para probar
# random_state=42 asegura que la división sea siempre la misma
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# Definimos los 4 modelos a comparar dentro de un diccionario
modelos = {
    # Regresión lineal clásica, sin transformaciones
    "Lineal": Pipeline([
        ("reg", LinearRegression())
    ]),
    # Regresión polinómica: crea combinaciones entre variables (ej: horas² , horas×asistencia)
    "Polinómico grado 2": Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("reg", LinearRegression())
    ]),


    # Ridge: penaliza coeficientes grandes pero no los elimina
    # alpha controla qué tan fuerte es la penalización
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),  # Ridge necesita que las variables estén en la misma escala
        ("ridge", Ridge(alpha=0.1))
    ]),
     # Lasso: penaliza coeficientes y puede llevar algunos exactamente a cero
    # (elimina variables poco importantes automáticamente)
    "Lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("lasso", Lasso(alpha=0.05))
    ])
}
# Entrenamos cada modelo y guardamos sus métricas
resultados = []
for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)   # Entrenamos con datos de entrenamiento
    pred = modelo.predict(X_test)   # Predecimos con datos de prueba (nuevos)
    resultados.append({
        "modelo": nombre,
        "RMSE_test": np.sqrt(mean_squared_error(y_test, pred)),   # Error promedio (menor = mejor)
        "R2_test": r2_score(y_test, pred)                # Qué tanto explica el modelo (mayor = mejor)

    })
# Mostramos la tabla ordenada del mejor al peor modelo según RMSE
pd.DataFrame(resultados).round(4).sort_values("RMSE_test")


Archivo encontrado en: rendimiento_estudiantil.csv


,modelo,RMSE_test,R2_test
1,Polinómico grado 2,2.9747,0.4058
3,Lasso,3.3151,0.2620
2,Ridge,3.3300,0.2554
0,Lineal,3.3301,0.2553


RIDGE:
- alpha=0.1  → coeficientes más grandes, modelo más flexible
- alpha=1.0  → balance intermedio
- alpha=10.0 → coeficientes más pequeños, modelo más restringido

LASSO:
- alpha=0.05 → casi todos los coeficientes sobreviven
- alpha=0.2  → algunos coeficientes se vuelven exactamente 0
- alpha=0.8  → más variables eliminadas automáticamente

## Conclusión

Escribe una respuesta corta:
¿cuándo convendría usar regularización en vez de una regresión lineal simple?


Conviene usar regularización cuando el modelo lineal simple está sobreajustado,
es decir, cuando aprende muy bien los datos de entrenamiento pero falla con datos nuevos.
También cuando tenemos muchas variables y queremos evitar que los coeficientes
se vuelvan demasiado grandes e inestables.